## Ferramenta Browser do AgentCore com Perfis de Navegador

Neste exemplo, você aprenderá como usar perfis de navegador dentro do AgentCore Browser. 
Este recurso permite persistir e reutilizar dados de perfil do navegador em múltiplas sessões. Um perfil de navegador armazena informações da sessão incluindo cookies e armazenamento local.

**Antes de começar, é importante que sua stack do CloudFormation tenha sido executada para implantar nosso e-commerce simples simulado que será usado neste tutorial.**

Nas saídas do CloudFormation, obtenha o nome da distribuição CloudFront:

![cfn_outputs](img/cfn_outputs.png)

Ou da saída do script [deploy.sh](sample-ecommerce/deploy.sh), obtenha o nome da distribuição CloudFront e preencha na próxima célula:

In [ ]:
CFN_URL="<sua-url-do-cloud-front>"

Para começar, instale as dependências e **reinicie seu kernel**:

In [ ]:
!pip install -qU -r requirements.txt

### 1. Criar um navegador personalizado

Nesta etapa, você declarará variáveis globais que serão usadas ao longo deste notebook.

In [ ]:
import boto3
import json
import sys
from botocore.exceptions import ClientError


sys.path.append('../helpers/')

iam_boto3 = boto3.client('iam')
s3 = boto3.client('s3')
browser_boto3 = boto3.client('bedrock-agentcore-control')
browser_cli = boto3.client('bedrock-agentcore')

session = boto3.Session()
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']
REGION = session.region_name

BROWSER_NAME = "browser_with_profiles"
BROWSER_PROFILE_NAME = "profile_sample"
BUCKET_NAME = f"ac-browser-demos-{ACCOUNT_ID}-{REGION}"
AC_ROLE_NAME = "ac-browser-execution-role"

#### 1.1 Criar um Bucket S3

Você precisa criar um Bucket S3, caso não exista, para armazenar gravações do navegador que baixaremos posteriormente.

In [ ]:
try:
    # verificar se o bucket existe
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} já existe")
except ClientError:
    # criar bucket
    create_params = {'Bucket': BUCKET_NAME}
    if REGION != 'us-east-1':
        create_params['CreateBucketConfiguration'] = {'LocationConstraint': REGION}
    s3.create_bucket(**create_params)
    print(f"Bucket {BUCKET_NAME} criado em {REGION}")

#### 1.2 Criar role IAM

Em seguida, você criará uma role IAM personalizada que será anexada ao AgentCore Browser:

In [ ]:
try: 
# Política de confiança
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }

    # Criar a role
    browser_role = iam_boto3.create_role(
        RoleName=AC_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy)
    )

    browser_role_arn = browser_role['Role']['Arn']

    print(f"Role ARN: {browser_role_arn}")

    # Política S3 para gravações
    ac_browser_policies = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "s3:PutObject",
                    "s3:GetObject",
                    "s3:ListBucket",
                    "s3:ListMultipartUploadParts",
                    "s3:AbortMultipartUpload"
                ],
                "Resource": [
                    f"arn:aws:s3:::{BUCKET_NAME}",
                    f"arn:aws:s3:::{BUCKET_NAME}/*"
                ]
            },
            {
                "Sid": "BedrockAgentCoreBrowserProfileUsageAccess",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:StartBrowserSession",
                    "bedrock-agentcore:SaveBrowserSessionProfile"
                ],
                "Resource": [
                    f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:browser-profile/{BROWSER_PROFILE_NAME}",
                    f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:browser-custom/{BROWSER_NAME}",
                ]
            }
        ]
    }

    # adicionar política inline S3
    iam_boto3.put_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyName='ac_custom_policies',
        PolicyDocument=json.dumps(ac_browser_policies)
    )

    # Anexar política gerenciada do Bedrock
    iam_boto3.attach_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyArn='arn:aws:iam::aws:policy/AmazonBedrockFullAccess'
    )

except ClientError as e:
    print(f'Exceção: {e}')
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        browser_role_arn = iam_boto3.get_role(RoleName=AC_ROLE_NAME)['Role']['Arn']
        print(f'Arn capturado: {browser_role_arn}')

Aguardar 10 segundos para garantir que a role IAM seja propagada

In [ ]:
import time

time.sleep(10)

#### 1.3 Criar o Browser personalizado do AgentCore

Você está criando um navegador personalizado para este exemplo, mas o recurso de Perfil de Navegador funciona também para o navegador gerenciado (aws.browser.v1).

In [ ]:
created_browser = browser_boto3.create_browser(
    name=BROWSER_NAME,
    executionRoleArn=browser_role_arn,
    networkConfiguration={
        'networkMode': 'PUBLIC'
    },
    recording={
        'enabled': True,
        's3Location': {
            'bucket': BUCKET_NAME,
            'prefix': 'browser_recordings/'
        }
    }
)

browser_id = created_browser['browserId']
print(f"Browser ID: {browser_id}")

#### 1.4 Criar o perfil de navegador

In [ ]:
created_profile = browser_boto3.create_browser_profile(
    name=BROWSER_PROFILE_NAME,
    description="Perfil de exemplo"
)

profile_id = created_profile['profileId']
print(f"Perfil criado: {profile_id}")

### 2. Teste

Para iniciar nosso teste, vamos iniciar uma nova sessão de navegador:

In [ ]:
response = browser_cli.start_browser_session(
    browserIdentifier=browser_id
)

session_id = response['sessionId']
print(f"Session ID: {session_id}")

Na célula seguinte, estamos assinando nossa requisição com Sigv4, para adicionar credenciais IAM nela.

In [ ]:
import browser_helper as helper

url = helper.get_url(browser_id, session_id)
headers = helper.get_signed_headers(url)
headers

#### 2.1 Testando em uma sessão

Agora, vamos usar o playwright para simular uma navegação no nosso e-commerce de exemplo.
[Playwright](https://playwright.dev/docs/intro) é um framework para Testes e Automação Web que é suportado pelo AgentCore Browser.
Então, para começar: 
1. Vamos navegar pela página *cart* para verificar que nosso carrinho está vazio
1. Vamos adicionar um echo dot ao nosso carrinho
1. Vamos verificar o carrinho novamente para ver que o produto foi adicionado.

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()
    
    try:
        # 1. Navegar para a página inicial
        await page.goto(
            f"{CFN_URL}/#home",
            wait_until="domcontentloaded"
        )

        await page.wait_for_timeout(2000)

        # 2. Adicionar primeiro item
        button = page.locator('button[onclick="addToCart(2)"]')
        await button.wait_for(state="visible")
        await button.click()

        await page.wait_for_timeout(2000)

        # 3. Adicionar segundo item
        button = page.locator('button[onclick="addToCart(4)"]')
        await button.wait_for(state="visible")
        await button.click()
        
        # 4. Verificar o carrinho
        view_cart_button = page.locator("#viewCart")
        await view_cart_button.wait_for(state="visible")
        await view_cart_button.click()
        await page.wait_for_timeout(2000)

        # 5. Retornar para a página inicial
        view_cart_button = page.locator("#backToProducts")
        await view_cart_button.wait_for(state="visible")
        await view_cart_button.click()

        # 6. Garantindo que o estado do carrinho será salvo localmente
        await page.evaluate("localStorage.setItem('cart', JSON.stringify(cart))")
        await page.wait_for_timeout(500)

    except Exception as error:
        print(f'Erro durante navegação: {error}')
        raise

#### 2.2 Salvar sessão no perfil

Agora, vamos salvar esta sessão no perfil que você criou.

In [ ]:
response = browser_cli.save_browser_session_profile(
    profileIdentifier=profile_id,
    browserIdentifier=browser_id,
    sessionId=session_id
)

print("Perfil salvo com sucesso")

#### 2.3 Encerrar sessão

Finalmente, vamos encerrar nossa sessão. 

In [ ]:
stoped_session = browser_cli.stop_browser_session(
    browserIdentifier=browser_id,
    sessionId=session_id
)
stoped_session

#### 2.4 Iniciar uma nova sessão

Agora, vamos iniciar uma nova sessão e adicionar nosso perfil de navegador, que tem nosso carrinho salvo.

In [ ]:
response = browser_cli.start_browser_session(
    browserIdentifier=browser_id,
    profileConfiguration={
        "profileIdentifier": profile_id
    }
)

session_id = response['sessionId']
print(f"Session ID: {session_id}")

In [ ]:
import browser_helper as helper

url = helper.get_url(browser_id, session_id)
headers = helper.get_signed_headers(url)
headers

#### 2.5 Verificar nosso carrinho

Finalmente, vamos ver que nosso produto já está selecionado no nosso carrinho.

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()
    
    try:
        # 1. Navegar para a página inicial
        await page.goto(
            f"{CFN_URL}/#home",
            wait_until="domcontentloaded"
        )

        await page.wait_for_timeout(5000)

        # 2. Verificar o carrinho
        view_cart_button = page.locator("#viewCart")
        await view_cart_button.wait_for(state="visible")
        await view_cart_button.click()
        await page.wait_for_timeout(2000)

    except Exception as error:
        print(f'Erro durante navegação: {error}')
        raise

Vamos encerrar nossa sessão

In [ ]:
stoped_session = browser_cli.stop_browser_session(
    browserIdentifier=browser_id,
    sessionId=session_id
)
stoped_session

### 3. Baixar gravação da sessão (Opcional)

Como você adicionou uma configuração de bucket no navegador, você tem os metadados que podem ser baixados e reproduzidos da navegação do navegador.

Então, vamos listar nossos arquivos do bucket S3 e key (a chave S3 será o ID da sessão):

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f'browser_recordings/{session_id}')

for obj in response.get('Contents', []):
    key = obj['Key']
    if key.endswith('.gz'):
        filename = key.split('/')[-1]  # Obter apenas o nome do arquivo
        s3.download_file(BUCKET_NAME, key, filename)
        print(f'Baixado: {filename}')

Em seguida, vamos abrir o arquivo zip e convertê-lo para um formato reproduzível.

In [ ]:
import gzip

# Descomprimir e ler eventos
events = []
with gzip.open(filename, 'rt') as f:
    for line in f:
        line = line.strip()
        if line:  # Pular linhas vazias
            events.append(json.loads(line))

# Salvar como JSON para rrweb
with open('events.json', 'w') as f:
    json.dump(events, f)


Finalmente, vamos reproduzi-lo:

In [ ]:
from IPython.display import HTML
import json

# Carregar eventos
with open('events.json', 'r') as f:
    events = json.load(f)

# Criar HTML com eventos inline
html = f"""
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/rrweb-player@latest/dist/style.css"/>
<div id="player"></div>
<script src="https://cdn.jsdelivr.net/npm/rrweb-player@latest/dist/index.js"></script>
<script>
    new rrwebPlayer({{
        target: document.getElementById('player'),
        props: {{ events: {json.dumps(events)} }}
    }});
</script>
"""

HTML(html)

### 4. Limpeza (Opcional)

Excluir o navegador AgentCore personalizado e o Perfil

In [ ]:
browser_boto3.delete_browser(browserId=browser_id)

In [ ]:
browser_boto3.delete_browser_profile(profileId=profile_id)